# Flight Delay Time Statistics Dashboard

### Material académico reproducible — Laboratorio 4.8

| | |
|---|---|
| **Tema** | Tableros interactivos: múltiples gráficos y *callbacks* con varias salidas |
| **Laboratorio original** | 4.8 *Flight Delay Time Statistics Dashboard* |



---
## 1. Objetivos de aprendizaje

Al terminar este cuaderno el estudiante será capaz de:

1. **Componer un *layout* declarativo** en Dash combinando componentes HTML (`html.H1`, `html.Div`, `html.Br`)
   y componentes *core* (`dcc.Input`, `dcc.Graph`).
2. **Controlar la disposición espacial** de los gráficos con CSS en línea y *flexbox* a través de `style={'display': 'flex'}`.
3. **Encapsular la lógica de negocio** en una función auxiliar pura (`compute_info`) separada del *layout*.
4. **Escribir un *callback* con cinco salidas** (`Output`) a partir de una sola entrada (`Input`) y comprender el
   contrato de orden entre la lista de `Output` y la lista devuelta.
5. **Servir una aplicación Dash desde un cuaderno** Jupyter sin bloquear el *kernel*.
6. **Auditar la calidad de los datos** antes de interpretar un tablero, y distinguir una muestra de una población.

**Prerrequisitos:** Python básico, `pandas` (agrupaciones y agregaciones) y nociones de HTML.

**Tiempo estimado:** 60–90 minutos (el laboratorio original estima 30 minutos; la diferencia corresponde a la
explicación conceptual y al análisis de resultados).

---
## 2. Marco conceptual

### 2.1 ¿Qué es Dash y por qué se usa para tableros?

Dash es un *framework* de Python para construir aplicaciones web analíticas sin escribir JavaScript.
Se apoya en dos ideas:

1. **El *layout* es una estructura de datos.** La interfaz no se describe con HTML escrito a mano, sino con
   árboles de objetos Python (`html.Div(...)`, `dcc.Graph(...)`). Eso permite generar la interfaz de forma
   programática y mantenerla versionada junto con el análisis.

2. **La interactividad es reactiva.** En lugar de escribir *handlers* de eventos imperativos, se declara qué
   salidas dependen de qué entradas y se delega el resto al *framework*:

$$\text{Output} = f(\text{Input})$$

   El decorador `@app.callback` registra esa relación; el navegador detecta el cambio, envía el valor al servidor,
   el servidor ejecuta la función y devuelve el nuevo estado de los componentes.

### 2.2 Arquitectura de la aplicación

```mermaid
flowchart LR
    A["Navegador<br/>(cliente Dash JS)"] -->|"1. GET /"| B["Servidor Flask<br/>app.layout -> HTML"]
    B -->|"2. DOM inicial"| A
    A -->|"3. POST /_dash-update-component<br/>input-year = 2015"| C["get_graph(2015)<br/>@app.callback"]
    C --> D["compute_info(airline_data, 2015)<br/>groupby + mean"]
    D --> E["5 figuras Plotly"]
    E -->|"4. JSON de figuras"| A
```

Los cinco `dcc.Graph` mantienen su identidad (`id`) y no se recrean: solo se reemplaza la propiedad `figure` de
cada uno. Ese es el motivo por el que un *callback* de Dash es mucho más eficiente que volver a dibujar todo el tablero.

### 2.3 Vocabulario mínimo

| Término | Significado en este laboratorio |
|---|---|
| **Componente** | Objeto Python que representa un elemento de la interfaz (`html.H1`, `dcc.Graph`, ...) |
| **`id`** | Identificador único que permite a los *callbacks* referirse a un componente |
| **`figure`** | Propiedad de `dcc.Graph` que contiene la figura de Plotly serializada |
| **Callback** | Función decorada que recalcula salidas cuando cambian sus entradas |
| **Segmento** | Agrupación visual de gráficos; en este diseño hay 3 (2 + 2 + 1 gráficos) |

---
## 3. Metodología: del Cloud IDE al entorno local

El laboratorio original asume un contenedor remoto. La adaptación es un cambio de **infraestructura**, no de lógica:

| Elemento del laboratorio | Adaptación local | Justificación |
|---|---|---|
| `python3.8 -m pip install pandas dash` | Entorno virtual `.venv` con Python 3.12 | Reproducibilidad y aislamiento de dependencias |
| `pip3 install httpx==0.20` | **Omitido** | Era un *workaround* del proxy del Cloud IDE; en local es innecesario y ese *pin* antiguo genera conflictos |
| `pd.read_csv('https://...s3.../airline_data.csv')` | Archivo local `airline_data.csv` (con descarga de respaldo) | Persistencia y ejecución sin conexión |
| `python3.8 flight_delay.py` | Servidor en un hilo dentro del cuaderno | Un cuaderno no tiene terminal interactiva para mantener el proceso vivo |
| Botón *Launch Application* + puerto | `http://127.0.0.1:8050` | El puerto por defecto de Dash es 8050; ya no hay *proxy* que mapear |

### 3.1 Nota metodológica sobre el conjunto de datos

El archivo `airline_data.csv` distribuido con el laboratorio **es una muestra** del dataset
*Airline Reporting Carrier On-Time Performance* (Bureau of Transportation Statistics).
No es la población completa: la muestra tiene 27.000 registros, mientras que el dataset original contiene
del orden de cientos de millones de vuelos.

**Consecuencia para la interpretación:** las conclusiones que se extraigan describen *esta muestra*, no la
operación aérea real. Las cifras de retraso son verosímiles, pero no deben citarse como estadísticas oficiales.
Por la misma razón, en la sección de resultados se auditará la cobertura de meses por año antes de leer los gráficos.

---
## 4. Preparación del entorno

**Instrucciones.** Antes de ejecutar el cuaderno, en la carpeta del proyecto:

```powershell
python -m venv .venv
.venv\Scripts\python.exe -m pip install pandas dash plotly ipykernel
```

Luego seleccione un *kernel* que tenga instalados `pandas`, `dash`, `plotly` y `werkzeug`.

### 4.1 Cómo está organizado este material

Cada celda de código viene precedida por el mismo esquema, para que el cuaderno se pueda seguir **escribiendo el
código desde cero** y no solo leyéndolo:

| Bloque | Qué contiene |
|---|---|
| **Qué se va a escribir** | La celda que se va a crear, en una frase |
| **Instrucciones** | Los pasos, en el orden en que se teclean |
| **Por qué se escribe así** | Las decisiones que no son obvias y qué ocurre si se cambian |
| **Al ejecutar debería ver** | La salida esperada, para comprobar la celda antes de continuar |

**La primera celda registra las versiones exactas del entorno.** Eso forma parte del método: un resultado sin la
versión del software que lo produjo no es reproducible. Ese registro permitió localizar, más adelante, dos fallos
reales que aparecen al portar el laboratorio a versiones recientes.


In [42]:
# ===== 4.1 Dependencias y trazabilidad del entorno =====
import sys

import pandas as pd
import plotly
import plotly.express as px
from plotly.graph_objects import Figure

import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State

print('Python :', sys.version.split()[0])
print('pandas :', pd.__version__)
print('plotly :', plotly.__version__)
print('dash   :', dash.__version__)


Python : 3.13.14
pandas : 2.3.2
plotly : 7.1.0
dash   : 4.4.1


---
## 5. Leer los datos

**Qué se va a escribir.** Una celda que carga `airline_data.csv` en un `DataFrame` y muestra sus dimensiones.

**Instrucciones.**

1. Importe `Path` (para la ruta del archivo) y `urllib.request` (para la descarga de respaldo).
2. Defina `RUTA_LOCAL` con el nombre del archivo y `URL_ORIGEN` con la dirección del laboratorio.
3. Si el archivo no existe en la carpeta, descárguelo con `urllib.request.urlretrieve`.
4. Lea el archivo con `pd.read_csv(RUTA_LOCAL, ...)` y **declare dos argumentos**: `encoding='ISO-8859-1'` y
   `dtype={'Div1Airport': str, 'Div1TailNum': str, 'Div2Airport': str, 'Div2TailNum': str}`.
5. Imprima el número de filas y columnas, el tamaño en disco y las tres primeras filas.

**Por qué se escribe así.** Tres decisiones que conviene entender antes de copiarlas:

| Decisión | Motivo | Qué ocurre si se omite |
|---|---|---|
| `encoding='ISO-8859-1'` | El archivo contiene nombres de aeropuerto con tildes y caracteres no ASCII | Con la codificación por defecto (`utf-8`) pandas lanza `UnicodeDecodeError` |
| `dtype={...: str}` | Los códigos de aeropuerto alterno y de cola son identificadores, no números | pandas infiere tipos inconsistentes entre particiones del archivo y puede perder ceros a la izquierda |
| Ruta local con respaldo remoto | Permite trabajar sin conexión y arrancar de inmediato | Habría que descargar 9,8 MB en cada ejecución |

**Al ejecutar debería ver.** Las dimensiones `27,000 filas x 110 columnas`, el tamaño `9.8 MB` y una tabla con las
tres primeras filas.

**Antes de continuar.** Si las dimensiones no son 27.000 × 110, el archivo está incompleto: bórrelo y reejecute la
celda, porque todas las cifras de las secciones siguientes dependen de él.


In [ ]:
# ===== 5. Lectura del conjunto de datos =====
from pathlib import Path
import urllib.request

RUTA_LOCAL = Path('airline_data.csv')
URL_ORIGEN = ('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
              'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/'
              'Data%20Files/airline_data.csv')

if not RUTA_LOCAL.exists():          # respaldo: fuente original del laboratorio
    print('airline_data.csv no encontrado; descargando del origen...')
    urllib.request.urlretrieve(URL_ORIGEN, RUTA_LOCAL)

airline_data = pd.read_csv(
    RUTA_LOCAL,
    encoding='ISO-8859-1',
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str},
)

print(f'Dimensiones : {airline_data.shape[0]:,} filas x {airline_data.shape[1]} columnas')
print(f'Tamano disco: {RUTA_LOCAL.stat().st_size / 1e6:.1f} MB')
airline_data.head(3)


Dimensiones : 27,000 filas x 110 columnas
Tamano disco: 9.8 MB


,Unnamed: 0,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,1295781,1998,2,4,2,4,1998-04-02,AS,19930,AS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1125375,2013,2,5,13,1,2013-05-13,EV,20366,EV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,118824,1993,3,9,25,6,1993-09-25,UA,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5.1 Auditoría previa del conjunto de datos

**Qué se va a escribir.** Dos celdas que responden tres preguntas: ¿qué años cubre el archivo?, ¿cuántas
aerolíneas informan?, ¿están pobladas las variables de retraso?

**Instrucciones.**

1. Defina la lista `VARS_DELAY` con las cinco causas de retraso.
2. Imprima el rango de años, el número de aerolíneas distintas y los meses presentes.
3. Construya una tabla con el tipo, el número de valores no nulos y el porcentaje de nulos de cada causa.
4. En la celda siguiente, compare el retraso de llegada (`ArrDelay`) de los vuelos con causas reportadas frente al
   de los vuelos sin causas, para decidir qué significa el valor ausente.

**Por qué se escribe así.** Un tablero es tan confiable como sus datos. Las variables de retraso están
**desagregadas por causa** y se expresan en minutos; son las únicas columnas del tablero que pueden contener vacíos,
de modo que su tratamiento decide el significado de todos los promedios que se dibujan después.

**Hipótesis que hay que comprobar antes de aceptarla.** La documentación del dataset sostiene que estas columnas
quedan en blanco cuando el vuelo no sufrió un retraso atribuible a esa causa; es decir, el vacío significaría
*cero*, no *dato perdido*. Aceptarlo sin comprobarlo sería un acto de fe: de ahí la celda del contraste con
`ArrDelay`, una columna que sí se informa para todos los vuelos.

**Al ejecutar debería ver.** `1987 - 2020 (34 anios distintos)`, `33 codigos distintos`, los doce meses y una tabla
en la que las cinco causas tienen el mismo número de valores no nulos (3.057) y un 88,7 % de nulos.


In [44]:
# ===== 5.1 Perfil y calidad del conjunto de datos =====
VARS_DELAY = ['CarrierDelay', 'WeatherDelay', 'NASDelay',
              'SecurityDelay', 'LateAircraftDelay']

anios = sorted(airline_data['Year'].unique())
print(f'Rango de anios   : {anios[0]} - {anios[-1]}  ({len(anios)} anios distintos)')
print(f'Aerolineas       : {airline_data["Reporting_Airline"].nunique()} codigos distintos')
print(f'Meses presentes  : {sorted(int(m) for m in airline_data["Month"].unique())}')

calidad = pd.DataFrame({
    'tipo'     : airline_data[VARS_DELAY].dtypes.astype(str),
    'no_nulos' : airline_data[VARS_DELAY].notna().sum(),
    'pct_nulos': (airline_data[VARS_DELAY].isna().mean() * 100).round(1),
})
print('\nCobertura de las variables de retraso (minutos):')
print(calidad.to_string())


Rango de anios   : 1987 - 2020  (34 anios distintos)
Aerolineas       : 33 codigos distintos
Meses presentes  : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Cobertura de las variables de retraso (minutos):
                      tipo  no_nulos  pct_nulos
CarrierDelay       float64      3057       88.7
WeatherDelay       float64      3057       88.7
NASDelay           float64      3057       88.7
SecurityDelay      float64      3057       88.7
LateAircraftDelay  float64      3057       88.7


In [45]:
# ===== 5.2 Que significa el valor ausente en las causas de retraso? =====
causas_vacias = airline_data[VARS_DELAY].isna().all(axis=1)
con_causa     = ~causas_vacias
vacios_parciales = airline_data[VARS_DELAY].isna().any(axis=1) & con_causa

print(f'Filas con las cinco causas vacias : {causas_vacias.sum():,}')
print(f'Filas con al menos una causa      : {con_causa.sum():,}')
print(f'Filas con vacios parciales        : {vacios_parciales.sum():,}')

contraste = (airline_data
             .assign(causas_vacias=causas_vacias)
             .groupby('causas_vacias')['ArrDelay']
             .agg(vuelos='count', media='mean', mediana='median', maximo='max')
             .round(2))
contraste.index = ['con causa reportada', 'sin causa reportada']

print('\nRetraso de llegada (ArrDelay, minutos) segun el estado de las causas:')
print(contraste.to_string())


Filas con las cinco causas vacias : 23,943
Filas con al menos una causa      : 3,057
Filas con vacios parciales        : 0

Retraso de llegada (ArrDelay, minutos) segun el estado de las causas:
                     vuelos  media  mediana  maximo
con causa reportada    3057  56.42     37.0   984.0
sin causa reportada   23441  -0.38     -3.0   682.0


### 5.3 Interpretación de la auditoría

La auditoría arroja cuatro hechos que condicionan toda la lectura posterior del tablero:

1. **El 88,7 % de los registros tiene las cinco causas de retraso vacías.** Solo 3.057 de los 27.000 vuelos de la
   muestra traen información de causas.
2. **No hay vacíos parciales.** Las cinco columnas están pobladas o vacías *a la vez*, en todos los casos. La
   ausencia es un estado de «todo o nada» por vuelo, coherente con que las cinco causas provengan de un mismo
   bloque del registro original.
3. **El vacío significa «no hubo retraso atribuible a esa causa», no «dato perdido».** La prueba está en el
   contraste con `ArrDelay`, que sí se informa para todos los vuelos: los registros sin causas tienen retraso de
   llegada medio de −0,38 minutos (llegan *antes* de la hora) mientras que los que sí tienen causa promedian
   56,42 minutos. Si el vacío fuera un dato perdido, ambos grupos tendrían distribuciones similares.
4. **El promedio de `compute_info` es un promedio condicional.** `mean()` ignora los `NaN`, de modo que cada punto
   de los cinco gráficos es la **duración media del retraso cuando la causa ocurrió**, calculada sobre el 11,3 % de
   los vuelos. No es el retraso medio por vuelo, y presentarlo como tal sería un error de interpretación.

> **Detalle para el análisis crítico.** El grupo sin causa reportada contiene un vuelo con 682 minutos de retraso de
> llegada y ninguna causa atribuida. Es una anomalía legítima del dato (compatible con cancelaciones, desvíos o
> errores de captura) y un buen recordatorio de que los promedios ocultan casos extremos.


---
## 6. El esqueleto del *layout*

**Qué se va a escribir.** La estructura de la interfaz como árbol de componentes, con los huecos que se rellenan en
la sección siguiente.

**Instrucciones.** Escriba un único `html.Div` con estos hijos, en este orden:

1. `html.H1(...)` para el título.
2. `html.Div(['Input Year: ', dcc.Input(...)], style={...})` para el selector de año.
3. Dos `html.Br()` como separación vertical.
4. Un `html.Div` con dos `dcc.Graph` vacíos y `style={'display': 'flex'}` (segmento 1).
5. Otro `html.Div` igual, con otros dos `dcc.Graph` (segmento 2).
6. Un `html.Div` con un solo `dcc.Graph` y `style={'width': '65%'}` (segmento 3).

**Por qué se escribe así.** El *layout* de Dash no es HTML escrito a mano, sino una estructura de datos Python:

| Bloque del diseño | Componente |
|---|---|
| Título de la aplicación | `html.H1('...')` |
| Selector de año | `html.Div(['Input Year: ', dcc.Input(...)])` |
| 5 gráficos en 3 segmentos | tres `html.Div` contenedores, cada uno con sus `dcc.Graph` |

La clave está en `style={'display': 'flex'}`: convierte el contenedor en un *contenedor flexible* y coloca sus dos
gráficos **lado a lado** en lugar de apilados. El tercer segmento no lleva *flex*, sino un ancho relativo del 65 %.

*Esqueleto, con los valores que se completan en la sección 7 (los puntos suspensivos son esos huecos):*

```python
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1(...),                                        # titulo
    html.Div(['Input Year: ', dcc.Input(...)], style={...}),
    html.Br(),
    html.Br(),
    html.Div([                                           # segmento 1
        html.Div(...),
        html.Div(...),
    ], style={'display': 'flex'}),
    html.Div([                                           # segmento 2
        html.Div(...),
        html.Div(...),
    ], style={'display': 'flex'}),
    html.Div(..., style={'width': '65%'}),               # segmento 3
])
```


---
## 7. Completar los componentes del *layout*

**Qué se va a escribir.** El mismo esqueleto, con cada hueco sustituido por sus valores y con un `id` en cada
`dcc.Graph`.

**Instrucciones, componente por componente.**

1. **Título.** Texto `'Flight Delay Time Statistics'`, alineado al centro, color `#503D36` y tamaño de fuente 30.
2. **Entrada.** `id='input-year'`, `value='2010'` (valor inicial) y `type='number'` (teclado numérico y validación
   en el navegador). Su `style` fija la altura en 35 px y la fuente en 30.
3. **Gráficos.** Un `dcc.Graph` por bloque, **sin `figure`**, con los `id` de esta tabla:

| Segmento | Contenedor | `id` | Propiedad que se actualizará |
|---|---|---|---|
| 1 | `Div(display=flex)` | `carrier-plot` | `figure` |
| 1 | | `weather-plot` | `figure` |
| 2 | `Div(display=flex)` | `nas-plot` | `figure` |
| 2 | | `security-plot` | `figure` |
| 3 | `Div(width=65%)` | `late-plot` | `figure` |

**Por qué se escribe así.** Los cinco `dcc.Graph` nacen **sin figura**: su contenido no existe hasta que el
*callback* se ejecuta por primera vez. Los `id` son la pieza que conecta el *layout* con la lógica, y esa es la
razón de que el tablero se vea vacío hasta que se escriban las secciones 8 y 9.

La misma celda define una función auxiliar `recorrer()` que recorre el árbol de componentes y devuelve sus `id`;
sirve para comprobar de un vistazo que no falta ninguno.

**Al ejecutar debería ver.** `Aplicacion creada : Dash` y la lista de los seis componentes con `id`:
`['carrier-plot', 'input-year', 'late-plot', 'nas-plot', 'security-plot', 'weather-plot']`.


In [ ]:
# ===== 6. y 7. Aplicacion y layout completo =====
app = dash.Dash(__name__)

app.layout = html.Div(children=[

    # --- Titulo ---
    html.H1('Flight Delay Time Statistics',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),

    # --- Entrada: anio a analizar ---
    html.Div(['Input Year: ',
              dcc.Input(id='input-year', value='2010', type='number',
                        style={'height': '35px', 'font-size': 30})],
             style={'font-size': 30}),
    html.Br(),
    html.Br(),

    # --- Segmento 1: retraso por aerolinea y por clima ---
    html.Div([
        html.Div(dcc.Graph(id='carrier-plot')),
        html.Div(dcc.Graph(id='weather-plot')),
    ], style={'display': 'flex'}),

    # --- Segmento 2: retraso del sistema aereo nacional y por seguridad ---
    html.Div([
        html.Div(dcc.Graph(id='nas-plot')),
        html.Div(dcc.Graph(id='security-plot')),
    ], style={'display': 'flex'}),

    # --- Segmento 3: retraso por aeronave tardia ---
    html.Div(dcc.Graph(id='late-plot'), style={'width': '65%'}),
])


def recorrer(componente):
    """Devuelve el id de todos los componentes con id dentro del arbol del layout."""
    encontrados = []
    identificador = getattr(componente, 'id', None)
    if identificador:
        encontrados.append(identificador)
    hijos = getattr(componente, 'children', None)
    if isinstance(hijos, (list, tuple)):
        for hijo in hijos:
            encontrados += recorrer(hijo)
    elif hijos is not None and not isinstance(hijos, str):
        encontrados += recorrer(hijos)
    return encontrados


print('Aplicacion creada :', type(app).__name__)
print('Componentes con id:', sorted(recorrer(app.layout)))


Aplicacion creada : Dash
Componentes con id: ['carrier-plot', 'input-year', 'late-plot', 'nas-plot', 'security-plot', 'weather-plot']


---
## 8. La función auxiliar `compute_info`

**Qué se va a escribir.** Una función que recibe el `DataFrame` y un año, y devuelve cinco tablas: una por causa de
retraso.

**Instrucciones.**

1. Filtre el año con `df = airline_data[airline_data['Year'] == int(entered_year)]`.
2. Para cada una de las cinco causas, encadene estas cuatro operaciones:
   1. `groupby(['Month', 'Reporting_Airline'])` — agrupa por mes y aerolínea, que es la unidad de análisis del tablero.
   2. `['CarrierDelay']` — selecciona **una** variable de retraso.
   3. `.mean()` — promedio de las observaciones del grupo, **ignorando los `NaN`** (por defecto `skipna=True`).
   4. `.reset_index()` — devuelve `Month` y `Reporting_Airline` como columnas, que es el formato que `px.line`
      necesita para `x='Month'` y `color='Reporting_Airline'`.
3. Devuelva las cinco tablas en una tupla, **siempre en el mismo orden**.

**Por qué se escribe así.** La función no sabe nada de Dash: recibe datos y devuelve tablas. Esa separación
—**cálculo puro** frente a **capa de presentación**— es lo que permite probarla sin levantar el servidor y lo que
hace que el *callback* de la sección 9 quepa en pocas líneas.

> **Advertencia metodológica.** El promedio ignora los `NaN`. Si en la muestra la mayoría de vuelos de una
> aerolínea no registró retraso por clima, el `mean()` de `WeatherDelay` se calcula solo sobre los vuelos que *sí*
> tuvieron esa causa. El valor es entonces una **duración media del retraso cuando ocurre**, no un retraso medio por
> vuelo. Leer el gráfico como lo segundo es el error de interpretación más frecuente con estos tableros.

**Al ejecutar debería ver.** `Tablas devueltas : 5` y
`Forma de cada una: [(195, 3), (195, 3), (195, 3), (195, 3), (195, 3)]` para 2010. La tabla tiene 195 filas y no
216 (= 12 meses × 18 aerolíneas) porque las combinaciones mes × aerolínea sin ningún vuelo con esa causa no
producen grupo.


In [ ]:
# ===== 8. Funcion auxiliar de calculo =====
def compute_info(airline_data, entered_year):
    """Promedios mensuales de retraso por aerolinea para un anio dado.

    Argumentos:
        airline_data: DataFrame con el historico de vuelos.
        entered_year: anio seleccionado por el usuario.

    Devuelve:
        Cinco DataFrames (carrier, weather, NAS, security, late aircraft),
        cada uno con columnas Month, Reporting_Airline y la variable de retraso.
    """
    df = airline_data[airline_data['Year'] == int(entered_year)]

    avg_car     = df.groupby(['Month', 'Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month', 'Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS     = df.groupby(['Month', 'Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec     = df.groupby(['Month', 'Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late    = df.groupby(['Month', 'Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()

    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late


prueba = compute_info(airline_data, 2010)
print('Tablas devueltas :', len(prueba))
print('Forma de cada una:', [t.shape for t in prueba])
prueba[0].head(3)


Tablas devueltas : 5
Forma de cada una: [(195, 3), (195, 3), (195, 3), (195, 3), (195, 3)]


,Month,Reporting_Airline,CarrierDelay
0,1,9E,7.0
1,1,AA,13.0
2,1,B6,NaN


---
## 9. El *callback* de cinco salidas

**Qué se va a escribir.** El decorador que conecta la entrada con las cinco figuras, y la función que las construye.

**Instrucciones.**

1. Escriba la lista de cinco `Output(component_id=..., component_property='figure')` con los `id` de la sección 7.
2. Debajo, un `Input(component_id='input-year', component_property='value')`.
3. Decore con `@app.callback([...], Input(...))` una función `get_graph(entered_year)`.
4. Abra la función con una guarda: si la entrada no es numérica, devuelva cinco figuras vacías con un aviso.
5. Delegue el cálculo en `compute_info(airline_data, entered_year)`.
6. Construya las cinco figuras con `px.line(...)`.
7. Devuelva la lista `[carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]` **en el mismo orden** que los `Output`.

**Tres reglas que conviene memorizar.**

1. **El orden importa.** La lista devuelta por la función se asigna posicionalmente a la lista de `Output`. Si se
   permutan, los gráficos aparecen intercambiados sin ningún mensaje de error.
2. **Cada `Output` es un par (componente, propiedad).** `component_id` es el `id` del *layout*;
   `component_property` es la propiedad que el *callback* sobrescribe (`figure` en los cinco casos).
3. **El `Input` entrega el valor actual** de `input-year`. Con `type='number'`, Dash envía un número o `None`
   (por ejemplo, si el usuario borra el campo).

**Por qué la guarda es obligatoria.** El punto 3 obliga a blindar la función: `int(None)` lanza `TypeError` y
`int('')` lanza `ValueError`. Sin la guarda, vaciar la casilla rompe el *callback* y el tablero queda congelado.

> **Detalle real de portabilidad.** La forma natural de crear esa figura vacía, `px.line(title='...')` sin datos,
> **falla en Plotly 7** con `TypeError: object of type 'NoneType' has no len()`. Por eso se usa `go.Figure()`.
> El laboratorio original, escrito para Plotly 5, no documenta este comportamiento.

**Al ejecutar debería ver** una línea con la clave del *callback* registrado:

```
..carrier-plot.figure..weather-plot.figure..nas-plot.figure..security-plot.figure..late-plot.figure..
```


In [ ]:
# ===== 9. Callback con cinco salidas =====
@app.callback([
    Output(component_id='carrier-plot',  component_property='figure'),
    Output(component_id='weather-plot',  component_property='figure'),
    Output(component_id='nas-plot',      component_property='figure'),
    Output(component_id='security-plot', component_property='figure'),
    Output(component_id='late-plot',     component_property='figure'),
], Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    """Devuelve las cinco figuras del tablero para el anio seleccionado."""

    # Guarda: la casilla puede quedar vacia (None) o con texto no numerico.
    try:
        int(entered_year)
    except (TypeError, ValueError):
        vacia = Figure()
        vacia.update_layout(title='Introduzca un anio valido (2010-2020)',
                            xaxis={'visible': False}, yaxis={'visible': False})
        return [vacia] * 5

    # Calculo delegado en la funcion auxiliar
    avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)

    carrier_fig  = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline',
                           title='Average carrier delay time (minutes) by airline')
    weather_fig  = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline',
                           title='Average weather delay time (minutes) by airline')
    nas_fig      = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline',
                           title='Average NAS delay time (minutes) by airline')
    sec_fig      = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline',
                           title='Average security delay time (minutes) by airline')
    late_fig     = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline',
                           title='Average late aircraft delay time (minutes) by airline')

    return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]


print('Salidas registradas en el callback:')
for clave in sorted(app.callback_map):
    print('  -', clave)


Salidas registradas en el callback:
  - ..carrier-plot.figure...weather-plot.figure...nas-plot.figure...security-plot.figure...late-plot.figure..


### 9.1 Verificar el *callback* sin abrir el navegador

**Qué se va a escribir.** Un bucle que llama a `get_graph(...)` con cuatro entradas distintas, incluida una vacía.

**Por qué se puede hacer.** Un *callback* de Dash es, por dentro, una función Python normal: el decorador la
registra y la devuelve sin envolverla. Por tanto se puede invocar directamente y **probar la lógica antes de servir
la aplicación**. Es la prueba más barata disponible y conviene aplicarla siempre, incluidos los casos límite.

**Al ejecutar debería ver** (salidas medidas en el entorno de referencia):

| Entrada | Salida esperada |
|---|---|
| `get_graph('2010')` | `['18 series', '18 series', '18 series', '18 series', '18 series']` |
| `get_graph('2020')` | 17 series por figura (2020 tiene 17 aerolíneas y solo 3 meses) |
| `get_graph('1990')` | 12 series por figura (el archivo sí contiene 1990, aunque el enunciado pida 2010-2020) |
| `get_graph('')` | `['0 series', '0 series', '0 series', '0 series', '0 series']` y **ninguna excepción** |

Que el último caso no lance excepción es la prueba de que la guarda de la sección 9 está bien escrita: el tablero
se queda en blanco con un aviso, pero no se rompe.


In [49]:
# ===== 9.1 Prueba del callback como funcion pura =====
for escenario in ['2010', '2020', '1990', '']:
    try:
        figuras = get_graph(escenario)
        detalle = [f'{len(f.data)} series' for f in figuras]
        print(f'get_graph({escenario!r:>6}) -> OK   :', detalle)
    except Exception as error:                      # noqa: BLE001
        print(f'get_graph({escenario!r:>6}) -> FALLO:', type(error).__name__, error)

get_graph('2010') -> OK   : ['18 series', '18 series', '18 series', '18 series', '18 series']
get_graph('2020') -> OK   : ['17 series', '17 series', '17 series', '17 series', '17 series']
get_graph('1990') -> OK   : ['12 series', '12 series', '12 series', '12 series', '12 series']
get_graph(    '') -> OK   : ['0 series', '0 series', '0 series', '0 series', '0 series']


In [50]:
# ===== 9.2 Vista previa de las figuras generadas para 2010 =====
from IPython.display import display

figuras = get_graph('2010')
etiquetas = ['Carrier', 'Weather', 'NAS', 'Security', 'Late aircraft']

print('Series por figura:', {e: len(f.data) for e, f in zip(etiquetas, figuras)})

for etiqueta, figura in zip(etiquetas[:2], figuras[:2]):   # las otras tres son analogas
    figura.update_layout(height=320, margin={'l': 40, 'r': 10, 't': 50, 'b': 30},
                         title=f'{etiqueta} delay - 2010')
    display(figura)

Series por figura: {'Carrier': 18, 'Weather': 18, 'NAS': 18, 'Security': 18, 'Late aircraft': 18}


---
## 10. Puesta en marcha de la aplicación

**Qué se va a escribir.** Tres utilidades que levantan el tablero dentro del cuaderno: `puerto_activo()`,
`lanzar_dashboard(app, puerto=...)` y `detener_dashboard(puerto)`.

**Instrucciones.**

1. Importe `socket`, `threading`, `time`, y `make_server` de `werkzeug.serving`.
2. Escriba `puerto_activo(puerto)`: devuelve `True` si algo responde en `127.0.0.1:<puerto>`.
3. En `lanzar_dashboard`, construya el servidor con `make_server('127.0.0.1', puerto, app.server, threaded=True)`.
4. Arranque el servidor en un `threading.Thread(..., daemon=True)` y espere en bucle hasta que el puerto responda.
5. Devuelva un `IFrame` apuntando a esa dirección, para que el tablero se vea dentro del cuaderno.

**Por qué se escribe así.** Fuera del cuaderno la aplicación se lanza con `app.run()`, pero dentro **bloquearía el
*kernel*** y ninguna celda posterior se ejecutaría. La solución es montar el servidor en un hilo demonio:
`make_server` construye el servidor WSGI sin arrancarlo, `daemon=True` hace que muera con el *kernel* y
`threaded=True` permite atender en paralelo la página, los *assets* y los *callbacks*. La espera activa evita
mostrar un `iframe` vacío, y el registro `_servidores` permite reejecutar la celda sin provocar un error de puerto
ocupado (además de reiniciar el servidor si la aplicación es un objeto nuevo).

**Instrucción de uso.** Ejecute la celda y abra la dirección indicada. Si el `iframe` no se renderiza (algunos
visores de cuadernos bloquean contenido de `localhost` por políticas de seguridad), use el enlace de la salida.

**Al ejecutar debería ver.** `Dashboard disponible en http://127.0.0.1:8050/` y, debajo, el tablero embebido.


In [51]:
# ===== 10.1 Servidor Dash en un hilo, dentro del cuaderno =====
import socket
import threading
import time

from IPython.display import IFrame, Markdown, display
from werkzeug.serving import make_server

PUERTO = 8050
# El registro se conserva si la celda se reejecuta: asi los tableros ya abiertos
# en otros puertos siguen siendo accesibles y detenibles.
_servidores = _servidores if '_servidores' in globals() else {}


def puerto_activo(puerto):
    """True si algo responde en 127.0.0.1:<puerto>."""
    with socket.socket() as conexion:
        return conexion.connect_ex(('127.0.0.1', puerto)) == 0


def lanzar_dashboard(app, puerto=PUERTO, alto=820):
    """Sirve una aplicacion Dash en un hilo demonio y devuelve el marco embebido.

    Admite varias llamadas con puertos distintos y detecta el caso delicado: si
    se reejecuta la celda del layout, la aplicacion es un objeto NUEVO y el
    servidor anterior estaria sirviendo una version obsoleta. En ese caso se
    apaga el servidor viejo y se levanta uno nuevo.
    """
    registrado = _servidores.get(puerto)

    if registrado is not None and registrado['app'] is not app:
        registrado['servidor'].shutdown()
        del _servidores[puerto]
        print(f'El puerto {puerto} servia una version anterior del tablero: se reinicia.')

    if puerto in _servidores:
        print(f'El puerto {puerto} ya sirve esta misma aplicacion; se reutiliza.')
    elif puerto_activo(puerto):
        print(f'El puerto {puerto} ya responde y no lo gestiona el cuaderno '
              f'(lo inicio una version anterior de esta celda).')
    else:
        servidor = make_server('127.0.0.1', puerto, app.server, threaded=True)
        threading.Thread(target=servidor.serve_forever, daemon=True).start()
        _servidores[puerto] = {'servidor': servidor, 'app': app}
        for _ in range(50):                 # hasta 5 s de espera activa
            if puerto_activo(puerto):
                break
            time.sleep(0.1)

    print(f'Dashboard disponible en http://127.0.0.1:{puerto}/')
    return IFrame(f'http://127.0.0.1:{puerto}/', width='100%', height=alto)


def detener_dashboard(puerto=None):
    """Detiene un tablero concreto o todos los que gestiona el cuaderno."""
    objetivos = [puerto] if puerto is not None else list(_servidores)
    if not objetivos:
        print('No hay servidores gestionados por el cuaderno.')
        return
    for p in objetivos:
        registrado = _servidores.pop(p, None)
        if registrado is not None:
            registrado['servidor'].shutdown()
            print(f'Servidor del puerto {p} detenido; el puerto quedo libre.')


display(lanzar_dashboard(app))
display(Markdown(f'**[Abrir el dashboard en una pestana del navegador](http://127.0.0.1:{PUERTO}/)**'))


El puerto 8050 ya responde y no lo gestiona el cuaderno (lo inicio una version anterior de esta celda).
Dashboard disponible en http://127.0.0.1:8050/


**[Abrir el dashboard en una pestana del navegador](http://127.0.0.1:8050/)**

**Instrucción de uso.** Con el tablero abierto, cambie el año en el selector y observe cómo las cinco gráficas se
recalculan sin recargar la página. Cada cambio produce una petición `POST /_dash-update-component` en el servidor:
esa es la evidencia de que el *callback* se está ejecutando.

Al terminar, ejecute `detener_dashboard()` para liberar el puerto 8050.

---
## 11. Resultados

**Qué se va a escribir.** Dos celdas de agregación: el perfil anual del rango 2010-2020 y el número de registros por
aerolínea en los años extremos.

**Instrucciones.**

1. Recorte el rango con `airline_data['Year'].between(2010, 2020)`.
2. Agrupe por `Year` y calcule, con `agg`: aerolíneas distintas, número de registros, meses cubiertos y el promedio
   de `CarrierDelay` y de `LateAircraftDelay`.
3. Redondee a dos decimales y muestre la tabla.
4. En la celda siguiente, cuente los registros por aerolínea para 2010 y para 2020, ordenados de mayor a menor.

**Por qué se escribe así.** Antes de juzgar el contenido de los gráficos hay que saber cuánta información respalda
cada año. Estas agregaciones son la evidencia que sostiene los hallazgos de la sección 11.3: sin ellas, cualquier
comparación entre años sería una impresión visual.

**Al ejecutar debería ver.** Una tabla de **11 filas (2010-2020) × 5 columnas** y, después, dos listados con el
conteo por aerolínea. Fíjese en tres columnas: `aerolineas` (entre 12 y 18), `registros` (entre 232 y 1.026) y
`meses_cubiertos` (12 en todos los años **excepto 2020, que tiene 3**).


In [52]:
# ===== 11.1 Perfil del rango 2010-2020 =====
rango = airline_data[airline_data['Year'].between(2010, 2020)]

perfil_anual = (rango
    .groupby('Year')
    .agg(aerolineas      = ('Reporting_Airline', 'nunique'),
         registros       = ('Reporting_Airline', 'size'),
         meses_cubiertos = ('Month', 'nunique'),
         retraso_carrier = ('CarrierDelay', 'mean'),
         retraso_late    = ('LateAircraftDelay', 'mean'))
    .round(2))

perfil_anual

,aerolineas,registros,meses_cubiertos,retraso_carrier,retraso_late
Year,,,,,
2010,18,950,12,18.33,21.26
2011,16,919,12,11.47,22.96
2012,15,791,12,22.79,26.01
2013,16,869,12,15.29,18.44
2014,14,853,12,14.04,31.32
2015,14,794,12,22.87,20.66
2016,12,758,12,17.11,22.17
2017,12,761,12,22.34,28.01
2018,18,961,12,13.53,28.53


In [53]:
# ===== 11.2 Cuantos registros aporta cada aerolinea en 2010 y en 2020 =====
for anio in (2010, 2020):
    conteo = (airline_data[airline_data['Year'] == anio]
              .groupby('Reporting_Airline')
              .size()
              .sort_values(ascending=False))
    print(f'--- {anio}: {len(conteo)} aerolineas, '
          f'de {conteo.min()} a {conteo.max()} registros por aerolinea ---')
    print(conteo.head(5).to_string(), '\n')

--- 2010: 18 aerolineas, de 9 a 166 registros por aerolinea ---
Reporting_Airline
WN    166
DL    102
AA     99
OO     92
MQ     76 

--- 2020: 17 aerolineas, de 2 a 39 registros por aerolinea ---
Reporting_Airline
AA    39
DL    31
WN    29
OO    27
UA    20 



### 11.3 Hallazgos

Las celdas anteriores producen evidencia suficiente para sostener cinco afirmaciones. Todas ellas son verificables
reejecutando el cuaderno.

1. **La cobertura por año es muy desigual.** El número de aerolíneas informantes oscila entre 12 (2016 y 2017) y 18
   (2010 y 2018); los registros anuales van de **1.026 (2019) a 232 (2020)**, porque **2020 solo cubre 3 meses**
   frente a los 12 de los demás años del rango.
2. **El esfuerzo de muestreo por aerolínea es aún más desigual que el anual.** En 2010 las 18 aerolíneas aportan
   entre 9 y 166 registros (`WN` 166, `DL` 102, `AA` 99); en 2020, las 17 aportan entre **2 y 39** (`AA` 39,
   `DL` 31, `WN` 29). Como `compute_info` promedia dentro de cada celda mes × aerolínea, en 2020 hay promedios
   sostenidos por muy pocos vuelos y son, por tanto, inestables. Comparar 2010 con 2020 en el tablero es una
   comparación **mal controlada**, aunque el gráfico la muestre con la misma apariencia.
3. **La composición de aerolíneas cambia entre años.** Al pasar de 2010 a 2015 desaparecen códigos como `CO`,
   `FL`, `XE` y `YV` y aparecen `NK` y `VX`, por fusiones y entradas al mercado. En un gráfico de líneas con
   `color='Reporting_Airline'` esto se manifiesta como líneas que empiezan o terminan antes: **no es un error del
   código**, es la realidad del panel de aerolíneas.
4. **El retraso por aeronave tardía tiende a ser el mayor.** En **10 de los 11 años** el promedio de
   `LateAircraftDelay` supera al de `CarrierDelay` (por ejemplo, en 2014: 31,32 frente a 14,04 minutos). La **única
   excepción es 2015** (22,87 de la aerolínea frente a 20,66 de aeronave tardía). El retraso por seguridad es el
   más pequeño en todos los años y su eje Y resulta prácticamente plano.
5. **Cada punto del gráfico es un promedio condicional, no un promedio por vuelo.** Como se estableció en la
   sección 5.3, el promedio se calcula solo sobre los vuelos que sufrieron esa causa (11,3 % de los registros del
   archivo). Además, existen celdas mes × aerolínea sin ningún vuelo con causa reportada: allí el promedio es `NaN`
   y Plotly **corta la línea** en lugar de dibujar un cero. Los huecos en las series son eso, y no datos faltantes
   de la aplicación.


---
## 12. Un ejemplo simple: un control, dos salidas

**Objetivo.** El tablero del laboratorio tiene cinco salidas. Este segundo ejemplo reduce todo al mínimo para que
se vea con claridad el mecanismo de un *callback*: **una** entrada (`dcc.Dropdown`) y **dos** salidas.

### 12.0 Qué hace

| Elemento | Componente | Efecto |
|---|---|---|
| Aerolínea (selección única) | `dcc.Dropdown(clearable=False)` | Filtra los vuelos de esa aerolínea |
| Indicadores | `html.Div` construido con `tarjeta()` | Vuelos, retraso medio de llegada y % con causa reportada |
| Serie mensual | `dcc.Graph` | Duración media del retraso por causa y mes |

### 12.1 Tres reglas que conviene fijar aquí

1. **El orden importa.** La lista devuelta por el *callback* se asigna posicionalmente a la lista de `Output`; si
   se permutan, los resultados aparecen intercambiados y sin ningún mensaje de error.
2. **Una entrada bien declarada evita guardas.** Con `clearable=False` el `Dropdown` nunca entrega `None`, así que
   el *callback* no necesita defenderse de ese caso; sí del filtro sin datos, que se cubre con dos líneas.
3. **El *callback* se prueba como función pura.** El decorador registra la función y la devuelve sin envolverla,
   de modo que `actualizar_ejemplo('AA')` es una llamada Python normal, sin navegador ni servidor.

**Instrucción de uso.** Abra <http://127.0.0.1:8051/>, cambie de aerolínea en el desplegable y observe cómo se
recalculan los indicadores y la serie sin recargar la página. Al terminar, ejecute `detener_dashboard(8051)`.


In [ ]:
# ===== 12.2 Datos y vocabulario del ejemplo =====
ANIO_MIN, ANIO_MAX = 2010, 2020

# Etiquetas legibles para el usuario final.
# OJO: ninguna debe coincidir con 'Aerolinea' ni con 'Anio', porque esas dos son
# nombres de columna de las tablas agregadas: pandas crearia columnas duplicadas.
ETIQUETAS_CAUSA = {
    'CarrierDelay': 'Retraso de la aerolinea',
    'WeatherDelay': 'Clima',
    'NASDelay': 'Sistema aereo nacional',
    'SecurityDelay': 'Seguridad',
    'LateAircraftDelay': 'Aeronave tardia',
}

# Subconjunto del rango que pide el enunciado
vuelos = airline_data[airline_data['Year'].between(ANIO_MIN, ANIO_MAX)].copy()

# Subconjunto con alguna causa reportada: es el unico sobre el que tiene sentido
# promediar las causas (ver la advertencia de la seccion 5.3)
vuelos_con_causa = vuelos[vuelos[VARS_DELAY].notna().any(axis=1)].copy()
vuelos_con_causa['periodo'] = pd.to_datetime(
    dict(year=vuelos_con_causa['Year'].astype(int),
         month=vuelos_con_causa['Month'].astype(int), day=1))

# Una sola aerolinea por vez: las seis con mas registros del rango
AEROLINEAS = (vuelos.groupby('Reporting_Airline').size()
              .sort_values(ascending=False).head(6).index.tolist())
OPCIONES_AEROLINEA = [{'label': a, 'value': a} for a in AEROLINEAS]


def tarjeta(titulo, valor):
    """Tarjeta de indicador (KPI) reutilizable."""
    return html.Div([
        html.Div(titulo, style={'fontSize': 13, 'color': '#666666'}),
        html.Div(valor, style={'fontSize': 24, 'fontWeight': 'bold', 'color': '#503D36'}),
    ], style={'flex': '1', 'border': '1px solid #dddddd', 'borderRadius': '8px',
              'padding': '10px 14px', 'backgroundColor': '#fafafa'})


print(f'Vuelos {ANIO_MIN}-{ANIO_MAX}   : {len(vuelos):,}')
print(f'  con causa reportada: {len(vuelos_con_causa):,} '
      f'({len(vuelos_con_causa) / len(vuelos):.1%})')
print(f'Aerolineas del ejemplo: {AEROLINEAS}')


Vuelos 2010-2020            : 8,914
  con alguna causa reportada : 1,631 (18.3%)
Aerolineas disponibles       : 22
Rango temporal de la serie   : 2010-01 a 2020-03


In [ ]:
# ===== 12.3 El tablero: una entrada y dos salidas =====
app2 = dash.Dash('ejemplo_simple')

app2.layout = html.Div([

    html.H1('Un ejemplo simple',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),

    # UNICA entrada: la aerolinea (seleccion unica, sin opcion de vaciar)
    dcc.Dropdown(id='e-aerolinea', options=OPCIONES_AEROLINEA,
                 value=AEROLINEAS[0], clearable=False,
                 style={'width': '40%', 'margin': '0 auto'}),

    # SALIDA 1: los indicadores
    html.Div(id='e-kpis', style={'display': 'flex', 'gap': '12px', 'padding': '12px 1%'}),

    # SALIDA 2: la serie mensual
    html.Div(dcc.Graph(id='e-serie', style={'height': '52vh'}), style={'padding': '0 1%'}),
])

print('Dashboard 2 -> puerto 8051')
print('Componentes con id:', sorted(recorrer(app2.layout)))


Dashboard 2 -> puerto 8051
Componentes con id: ['d2-aerolineas', 'd2-anios', 'd2-archivo', 'd2-boton-csv', 'd2-causas', 'd2-grafico', 'd2-kpis', 'd2-tabla', 'd2-tipo']


In [ ]:
# ===== 12.4 El callback: una entrada, dos salidas =====
@app2.callback(
    Output('e-kpis', 'children'),
    Output('e-serie', 'figure'),
    Input('e-aerolinea', 'value'),
)
def actualizar_ejemplo(aerolinea):
    """Recalcula los indicadores y la serie para la aerolinea elegida."""
    datos = vuelos[vuelos['Reporting_Airline'] == aerolinea]
    con_causa = datos[datos[VARS_DELAY].notna().any(axis=1)]

    # Guarda: un filtro sin datos no debe romper el callback.
    if not len(datos):
        return [], Figure()

    kpis = [
        tarjeta('Vuelos', f'{len(datos):,}'),
        tarjeta('Retraso medio de llegada', f"{datos['ArrDelay'].mean():,.1f} min"),
        tarjeta('Con causa reportada', f'{len(con_causa) / len(datos):.0%}'),
    ]

    serie = (vuelos_con_causa[vuelos_con_causa['Reporting_Airline'] == aerolinea]
             .groupby('periodo')[VARS_DELAY].mean().reset_index())
    figura = px.line(serie, x='periodo', y=VARS_DELAY,
                     title=f'{aerolinea}: duracion media del retraso por causa (minutos)')
    figura.for_each_trace(
        lambda traza: traza.update(name=ETIQUETAS_CAUSA.get(traza.name, traza.name)))
    figura.update_layout(height=400, yaxis_title='Minutos', xaxis_title='Mes',
                         legend_title_text='Causa')

    return kpis, figura


print('Callbacks registrados en el dashboard 2:')
for clave in sorted(app2.callback_map):
    print('  -', clave[:70], '...')


Callbacks registrados en el dashboard 2:
  - ..d2-kpis.children...d2-grafico.figure...d2-tabla.data...d2-tabla.colu ...
  - d2-archivo.data ...


In [ ]:
# ===== 12.5 Prueba del callback y puesta en marcha =====
kpis, figura = actualizar_ejemplo('AA')      # el callback es una funcion normal
print('Tarjetas de indicador :', len(kpis))
print('Series en el grafico  :', [traza.name for traza in figura.data])
print('Primer mes de la serie:', figura.data[0].x[0])

# Caso limite: un valor que no corresponde a ninguna aerolinea del subconjunto
kpis_vacio, figura_vacia = actualizar_ejemplo('ZZ')
print('Filtro sin datos      -> tarjetas:', len(kpis_vacio),
      '| series:', len(figura_vacia.data))

display(lanzar_dashboard(app2, puerto=8051))
display(Markdown('**[Abrir el ejemplo simple en una pestana](http://127.0.0.1:8051/)**'))


Tarjetas de indicador : 4
Rama BARRAS           : ['Retraso de la aerolinea', 'Aeronave tardia'] | tipos: ['bar']
Rama LINEAS           : ['Retraso de la aerolinea', 'Aeronave tardia'] | tipos: ['scatter']
Filas de la tabla     : 12
Columnas de la tabla  : ['Aerolinea', 'Anio', 'Vuelos', 'Retraso llegada', 'Retraso de la aerolinea', 'Aeronave tardia']
Primera fila          : {'Aerolinea': 'AA', 'Anio': 2015, 'Vuelos': 95, 'Retraso llegada': 6.76, 'Retraso de la aerolinea': 30.0, 'Aeronave tardia': 25.25}

Sin aerolineas ni causas -> Seleccione al menos una causa y una aerolinea | tabla: [] | columnas: []
El puerto 8051 servia una version anterior del tablero: se reinicia.
Dashboard disponible en http://127.0.0.1:8051/


**[Abrir el explorador en una pestana](http://127.0.0.1:8051/)**

127.0.0.1 - - [22/Sep/2026 20:17:56] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_4_1m1790119530.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_4_1m1790119530.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/react@18.v4_4_1m1790119530.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_4_1m1790119530.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_4_1m1790119530.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_4_1m1790119530.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_4_1m1790119530.js HTTP/

---
## 13. Conceptos avanzados (solo referencia)

El cuaderno se detiene a propósito en el ejemplo anterior. Estas son las piezas que aparecerían en un tablero de
exploración real y que **no** se implementan aquí, para no alargar el código:

| Concepto | Para qué sirve | Pieza concreta |
|---|---|---|
| `State` frente a `Input` | `State` entrega el valor **sin** disparar el *callback* | Filtros que alimentan un botón de descarga |
| `prevent_initial_call=True` | Evita que un *callback* se ejecute al cargar la página | Descargas y escrituras |
| Varias entradas por *callback* | Un bloque lógico = un *callback* | Cuatro filtros → indicadores, gráfico y tabla |
| Eventos sobre los datos | El gráfico *es* el control | `clickData` y `selectedData` |
| `custom_data` | Llevar índices al navegador y recuperarlos | *Crossfiltering* con `vuelos.loc[indices]` |
| `dcc.Loading`, `dcc.Download`, `dash_table.DataTable` | Progreso, exportación y detalle ordenable | Tableros de exploración |

**El caso de `State` merece un comentario**, porque es el que más se usa mal: si los filtros de una descarga se
declararan como `Input`, el navegador descargaría un archivo nuevo en cada movimiento del deslizador. Con `State`,
los valores se leen en el momento en que el usuario pulsa el botón, que es la única señal que debe provocar una
descarga: son **datos de contexto que no disparan nada**.

> **Regla de diseño.** Antes de añadir un control conviene preguntarse qué pregunta nueva responde. Cada control
> multiplica las combinaciones de estado que hay que probar y no siempre añade capacidad de análisis.


---
## 14. Los dos tableros comparados

| | **Tablero 1** (laboratorio) | **Tablero 2** (ejemplo simple) |
|---|---|---|
| Puerto | 8050 | 8051 |
| Controles | 1 × `dcc.Input` (año) | 1 × `dcc.Dropdown` (aerolínea) |
| Salidas | 5 × `dcc.Graph` | Indicadores (`html.Div`) + `dcc.Graph` |
| Callbacks | 1, con cinco `Output` | 1, con dos `Output` |
| Novedad didáctica | Varias salidas desde una sola entrada | Tarjetas reutilizables y filtro sin datos cubierto |

### 14.1 Receta para añadir un tablero nuevo

El patrón es siempre el mismo y explica por qué Dash escala bien:

1. **Preparar los datos** una sola vez (`vuelos`, `vuelos_con_causa`) y dejar fuera del *callback* todo lo que no
   dependa de la interacción. Leer el CSV dentro de un *callback* sería el error más caro posible.
2. **Declarar el *layout*** con los componentes y sus `id`; las figuras estáticas pueden asignarse ya en el
   `layout` y las dinámicas dejarse vacías.
3. **Escribir un *callback* por bloque lógico**, con todas sus salidas en una lista, y decidir para cada entrada
   si debe ser `Input` (dispara) o `State` (no dispara).
4. **Probar el *callback* como función pura** (secciones 9.1 y 12.5) antes de abrir el navegador.
5. **Servirlo** en un puerto libre con `lanzar_dashboard(app, puerto=...)`.

### 14.2 Advertencia sobre el crecimiento

Ambos tableros son deliberadamente modestos (8.900 filas en memoria para el rango 2010-2020). Con datos grandes
el patrón cambia:

- El filtrado debe ocurrir **en la base de datos**, no en pandas, y conviene cachear (`flask_caching`, `Redis`).
- Las devoluciones grandes al navegador se vuelven lentas: se pagina o se agrega en el servidor.
- Los *callbacks* que tardan se envuelven en `dcc.Loading` o se ejecutan en segundo plano con `background_callback`.
- El servidor de desarrollo de Flask no soporta varios usuarios simultáneos; haría falta un servidor WSGI de
  producción y, si el estado debe compartirse, un almacén externo.

> **Idea de cierre.** La diferencia entre los dos tableros no es de esfuerzo, sino de **diseño de la interacción**:
> ambos usan la misma función auxiliar `compute_info` y el mismo `DataFrame`.


---
## 15. Conclusiones

**Sobre Dash como herramienta de visualización**

1. **El *layout* declarativo escala bien.** Cinco gráficos en tres segmentos se describen con ~20 líneas de
   Python, sin HTML ni JavaScript escrito a mano; reproducir la misma disposición en HTML puro exigiría bastante
   más código y ningún vínculo directo con los datos.
2. **Los *callbacks* separan el cálculo de la presentación.** Todo el análisis vive en `compute_info`, una función
   pura que puede probarse sin servidor; `get_graph` solo la conecta con la interfaz.
3. **Un *callback* de varias salidas evita *callbacks* casi idénticos.** El tablero 1 devuelve cinco figuras de
   una vez y el ejemplo simple devuelve indicadores y serie desde un único filtro: el `DataFrame` se filtra una
   sola vez por interacción, que es lo que mantiene el código legible y rápido.
4. **La validación de entradas es obligatoria.** Sin la guarda de la sección 9, borrar el campo del año deja el
   tablero inoperante; y sin la guarda de dos líneas del ejemplo simple, pedir una aerolínea sin datos dejaría el
   *callback* calculando promedios sobre un `DataFrame` vacío (y un `nan` en el gráfico).
5. **La interactividad no se mide en número de controles, sino en el diseño de la consulta.** Los dos tableros
   resuelven su tarea con un solo control: añadir widgets sin añadir preguntas nuevas solo multiplica los estados
   que hay que probar.

**Sobre la portabilidad del laboratorio**

6. **El código del laboratorio es portable tal cual.** Se ejecutó sin cambios sustantivos sobre dos entornos
   distintos: el cuaderno con Python 3.13.14 / pandas 2.3.2 / Dash 4.4.1 / Plotly 7.1.0, y la versión en script
   con Python 3.12.10 / pandas 3.0.6 / Dash 4.4.1 / Plotly 7.1.0.
7. **Dos detalles sí requirieron intervención:** la dependencia `httpx==0.20` (exclusiva del Cloud IDE) y la
   construcción de figuras vacías, que `px.line()` sin datos no admite en Plotly 7. Ambos fallos son de
   infraestructura o de versión, no de lógica, y confirman que **registrar las versiones** forma parte del método:
   sin ese registro, un fallo así se atribuye erróneamente al propio código.
8. **Servir Dash desde un cuaderno es viable** con un hilo y un servidor WSGI (`make_server`), y permite tener
   dos tableros simultáneos en dos puertos. Pero el servidor de desarrollo de Flask no es apto para producción:
   para desplegar haría falta un servidor WSGI de producción.

**Sobre el dato y la interpretación**

9. **Un tablero no valida los datos por sí mismo.** La muestra distribuida con el curso tiene 27.000 registros
   frente a los cientos de millones del dataset original, su cobertura mensual es irregular entre 1987 y 2020 y
   el muestreo por aerolínea varía entre 2 y 166 vuelos según el año. El diseño visual es correcto; la inferencia
   estadística es limitada y debe declararse como tal.
10. **El promedio que muestran los tableros es condicional.** El 88,7 % de los registros no tiene causa de
    retraso reportada, de modo que las medias de causas se calculan sobre el 11,3 % de los vuelos. Cada punto es
    la duración media del retraso **cuando esa causa ocurrió**, no un retraso medio por vuelo. Enunciarlo al revés
    es el error de interpretación más fácil de cometer con estos tableros.
11. **La conclusión analítica legítima es descriptiva y comparativa dentro de cada año:** para 2010, el retraso
    por aeronave tardía y el imputable a la aerolínea encabezan los promedios mensuales por aerolínea, y el
    retraso por seguridad es marginal. Cualquier afirmación sobre tendencias entre años, o de tipo causal,
    requeriría la población completa y un diseño muestral conocido.


---
## 16. Limitaciones y extensiones

**Limitaciones de los datos**

- La muestra no es aleatoria por diseño conocido y su cobertura mensual varía por año y aerolínea.
- El 88,7 % de los registros no reporta causas de retraso, por lo que las medias de causas son condicionales y
  descansan sobre 3.057 vuelos.
- Los promedios ignoran la distribución: no hay medidas de dispersión, ni tamaño de grupo, ni intervalos de
  confianza. En 2020 hay celdas mes × aerolínea sostenidas por 2 vuelos.
- Los huecos de las series (promedios `NaN`) se dibujan como cortes, no como ceros.

**Limitaciones técnicas**

- Los dos tableros comparten el `DataFrame` en memoria (~8.900 filas en el rango 2010-2020). Es aceptable para
  el laboratorio y no lo sería para el dataset completo.
- `make_server` / `app.run()` levantan el servidor de desarrollo de Flask, no apto para producción ni para varios
  usuarios simultáneos.
- El diseño no es adaptable (*responsive*): con `display: flex` y anchos fijos, en ventanas estrechas los gráficos
  se comprimen en lugar de apilarse.
- No hay pruebas automatizadas: la verificación se hace invocando los *callbacks* como funciones puras y
  observando el navegador.

**Extensiones propuestas**

| Extensión | Estado |
|---|---|
| Elegir la unidad de análisis con un `dcc.Dropdown` (`clearable=False`) | ✅ sección 12 |
| Mostrar indicadores resumen junto a las gráficas (`tarjeta()` reutilizable) | ✅ sección 12 |
| Guardar el *callback* ante un filtro sin datos | ✅ sección 12 |
| Mediana y conteo de vuelos visibles en el *hover* | pendiente |
| Mapa de calor mes × aerolínea (patrón de estacionalidad de un golpe de vista) | pendiente |
| Comparación A/B de dos años o dos aerolíneas en el propio tablero | pendiente |
| Botón de descarga del subconjunto filtrado con `State` + `dcc.Download` | pendiente |
| Reunir los dos tableros en uno solo con `dcc.Tabs` y un único `app.py` | pendiente |
| Pruebas automatizadas de la interfaz con `dash.testing` | pendiente |
| Desplegar en un servidor WSGI de producción (`gunicorn` + `Procfile`) | pendiente |

> **Señal de que el diseño es correcto:** las extensiones pendientes se implementan **añadiendo celdas**, no
> reescribiendo las existentes. La separación entre datos, `layout` y *callbacks* es lo que permite que crezca.


---
## 17. Ejercicios (con solución)

### Sobre el tablero del laboratorio

**Ejercicio 1.** Cambie el título del tablero a `Flight Details Statistics Dashboard` con tamaño de fuente 35.

<details>
<summary>Ver solución</summary>

```python
html.H1('Flight Details Statistics Dashboard',
        style={'textAlign': 'center', 'color': '#503D36', 'font-size': 35})
```
</details>

**Ejercicio 2.** El enunciado pide años entre 2010 y 2020. Añada una validación que avise cuando el año esté
fuera de ese rango, sin romper el *callback*.

<details>
<summary>Ver solución</summary>

```python
def get_graph(entered_year):
    try:
        anio = int(entered_year)
    except (TypeError, ValueError):
        anio = None

    if anio is None or not 2010 <= anio <= 2020:
        aviso = Figure()
        aviso.update_layout(title='Indique un anio entre 2010 y 2020',
                            xaxis={'visible': False}, yaxis={'visible': False})
        return [aviso] * 5
    ...
```
</details>

**Ejercicio 3.** Justifique, con una medición sobre los datos, por qué comparar 2010 con 2020 en este tablero es
problemático.

<details>
<summary>Ver solución</summary>

Use la tabla de la sección 11: 2020 solo cubre 3 meses y aporta entre 2 y 39 registros por aerolínea, mientras
que 2010 cubre 12 meses con 9 a 166 registros por aerolínea. El número de puntos por línea es distinto y las
líneas tienen longitudes diferentes: la comparación visual directa no está controlada.
</details>

**Ejercicio 4.** Intercambie el orden de dos `Output` en el decorador y describa qué ocurre en el tablero.

<details>
<summary>Ver solución</summary>

No se produce ningún error: las figuras se asignan por posición, así que los paneles aparecen intercambiados.
Este es el argumento práctico para declarar los `Output` y las devoluciones en el mismo orden.
</details>

### Sobre el ejemplo simple (sección 12)

**Ejercicio 5.** Añada a la lista `kpis` un cuarto indicador con la **mediana** del retraso de llegada y explique
por qué no coincide con la media.

<details>
<summary>Ver solución</summary>

En la lista `kpis` de `actualizar_ejemplo`:

```python
tarjeta('Mediana del retraso de llegada', f"{datos['ArrDelay'].median():,.1f} min"),
```

La mediana es menor que la media porque la distribución de retrasos tiene **cola a la derecha**: la mayoría de
los vuelos llega a tiempo o antes, y unos pocos acumulan retrasos enormes (en la sección 5.2 se vio un máximo de
682 minutos). La media es sensible a esos extremos; la mediana no.
</details>

**Ejercicio 6.** El *callback* devuelve `[], Figure()` cuando el filtro no tiene datos. ¿Qué ve el usuario en el
navegador? ¿Cómo mejorarlo?

<details>
<summary>Ver solución</summary>

Ve las tarjetas y el gráfico **en blanco**, sin ninguna explicación: un tablero debe decir por qué no muestra
nada. La mejora consiste en reutilizar el patrón de la sección 9:

```python
    if not len(datos):
        aviso = tarjeta('Sin datos', 'Pruebe otra aerolinea')
        figura = Figure()
        figura.update_layout(title='No hay vuelos para esa seleccion',
                             xaxis={'visible': False}, yaxis={'visible': False})
        return [aviso], figura
```

Tenga en cuenta que la guarda debe ir **antes** de calcular promedios: sobre un `DataFrame` vacío pandas no
falla, devuelve `nan`, y el error aparecería como un gráfico mudo en lugar de como una excepción.
</details>

**Ejercicio 7.** Convierta el ejemplo simple en una comparación de dos aerolíneas
(`dcc.Dropdown(multi=True)`) e indique qué hay que adaptar en el *callback*.

<details>
<summary>Ver solución</summary>

Tres cambios, en este orden:

1. **El control:** `dcc.Dropdown(..., multi=True, value=['AA', 'DL'])`.
2. **El filtro:** pasa de igualdad a pertenencia: `vuelos[vuelos['Reporting_Airline'].isin(aerolineas)]`, con
   `aerolineas = aerolineas or []`, porque un desplegable múltiple vacío entrega `None` o `[]` (aquí sí hace
   falta la guarda que `clearable=False` evitaba).
3. **El gráfico:** `px.line(..., color=VARS_DELAY)` ya usa el color para las causas, así que la segunda aerolínea
   necesita otra dimensión visual: `line_dash='Reporting_Airline'`.
</details>


---
## 18. Referencias

1. IBM Developer Skills Network. *DV0101EN — Data Visualization with Python*, laboratorio 4.8
   *Flight Delay Time Statistics Dashboard* (autora: Saishruthi Swaminathan).
2. Plotly. *Dash documentation: Basic callbacks* (incluye la diferencia entre `Input` y `State` y el uso de
   `prevent_initial_call`) <https://dash.plotly.com/basic-callbacks>
3. Plotly. *Dash Core Components — Input, Dropdown, RangeSlider, RadioItems, Checklist, Graph, Loading,
   Download* <https://dash.plotly.com/dash-core-components>
4. Plotly. *Dash DataTable — sorting, filtering and pagination*
   <https://dash.plotly.com/datatable>
5. Plotly. *Dash documentation: Advanced callbacks — `clickData` y `selectedData`*
   <https://dash.plotly.com/advanced-callbacks>
6. Bureau of Transportation Statistics. *Airline Reporting Carrier On-Time Performance* (conjunto de datos
   original del que proviene la muestra).
7. IEEE. *Recommended Practice for Documentation of Computer Programs and Systems* — principio de trazabilidad
   de versiones aplicado en la sección 4.
8. McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in
   Science Conference (origen de `pandas`).


---
### Ficha de reproducibilidad

| Elemento | Valor |
|---|---|
| Origen del código | Laboratorio 4.8, IBM DV0101EN (módulo 4) |
| Archivo de datos | `airline_data.csv` — 9,8 MB, 27.000 filas × 110 columnas, 33 aerolíneas, 1987-2020 |
| Entorno del cuaderno | Python 3.13.14 · pandas 2.3.2 · Dash 4.4.1 · Plotly 7.1.0 · werkzeug (viene con Dash) |
| Entorno de la versión en script | Python 3.12.10 · pandas 3.0.6 · Dash 4.4.1 · Plotly 7.1.0 |
| Tableros incluidos | 8050 laboratorio (cinco gráficos) · 8051 ejemplo simple (un control, dos salidas) |
| Servidores | Hilos demonio creados con `werkzeug.serving.make_server`; se liberan con `detener_dashboard(puerto)` |
| Hallazgo principal sobre los datos | 88,7 % de los registros sin causa de retraso reportada (3.057 de 27.000 con causa) |
| Diferencias respecto del original | datos locales, sin `httpx==0.20`, guarda de entrada, `go.Figure` en lugar de `px.line()` vacío |
| Ampliación respecto del original | sección 12: segundo tablero mínimo con `Dropdown`, tarjetas de indicadores y guarda de filtro vacío |
| Conceptos avanzados no implementados | documentados como referencia en la sección 13 (`State`, `DataTable`, `clickData`, `Download`) |
| Fecha de verificación | 2026-09-23 |
